In [1]:
# Setup & Connect to Database

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries loaded successfully!")

✅ All libraries loaded successfully!


In [3]:
# Load Data from MySQL
# WHAT: Connect to MySQL and pull data

# Import settings from config
import sys
sys.path.append(r'D:\Vaidehi Study\Regional_Sales_Analysis\04_scripts')
from config import DB_CONFIG

# Create connection
engine = create_engine(f"mysql+pymysql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}")

In [6]:
df_raw = pd.read_csv(
    r'D:\Vaidehi Study\Regional_Sales_Analysis\01_data\Sales_Orders.csv',
    dtype=str,
    usecols=range(12)
)
df_raw.columns = df_raw.columns.str.strip()

for col in ['Unit Price', 'Line Total', 'Total Unit Cost', 'Order Quantity']:
    df_raw[col] = pd.to_numeric(df_raw[col].str.replace(',', '', regex=False))

df_raw['OrderDate'] = pd.to_datetime(df_raw['OrderDate'], format='%m/%d/%Y')

print(df_raw['Unit Price'].max()) 
print(df_raw.shape)               

df_raw.to_sql('sales_orders', con=engine, if_exists='replace', index=False, chunksize=1000)

6566.0
(64104, 12)


64104

In [ ]:
# DATA IMPORT — Run once to load Sales Orders into MySQL
# All cleaning done in Python BEFORE loading to avoid MySQL silent truncation:
# - Commas stripped from Unit Price, Line Total, Total Unit Cost
# - OrderDate converted to proper datetime with explicit format
# - Only first 12 columns loaded (skips 2 unnamed extra columns from source file)
# Verified: max Unit Price = 6566.0 (confirms no comma truncation)
# Then push to MySQL via SQLAlchemy — faster and safer than Workbench wizard